# 实验六：网络损伤对高清视频的影响（Network Impairments on HD Video）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU（SciPy 缩放在大帧上稍慢）**

---

## 实验概述

网络传输对视频画质的损伤，是视频流媒体系统设计中最核心的挑战之一：无论上层的编码算法多么先进，一旦网络条件恶化，画面依然会出现各种可见的失真。理解不同网络损伤类型各自的视觉表现规律，是设计缓解方案（如前向纠错、抖动缓冲、自适应码率）的基础。

本实验对一张合成 HD 测试帧（540×960）以及一个更接近真实场景的风景画面，分别模拟三种典型的网络损伤——丢包（Packet Loss）、抖动（Jitter）、带宽骤降（Bandwidth Drop），通过全局对比、差异热力图、像素级放大等多种可视化手段，直观展示每种损伤独特的视觉特征。

本实验对应课程中「网络损伤与视频质量退化」部分的核心内容，也是理解实验七（VR 视频损伤对比）的基础。

## 学习目标

完成本实验后，你应该能够：

1. 区分丢包、抖动、带宽骤降三种网络损伤各自独特的视觉表现；
2. 解释每种损伤在真实传输链路中产生的技术原因；
3. 读懂差异热力图（Difference Map）和像素级放大对比，从中定量观察受损区域；
4. 说明三种损伤的严重程度与网络条件之间的对应关系；
5. 将实验观察到的失真现象与真实系统的缓解方案（ABR、FEC、抖动缓冲区）联系起来。

## 背景与基本原理

### 丢包（Packet Loss）

在基于 UDP 的实时视频传输中，数据包可能因网络拥塞、路由问题等原因丢失，接收端无法获得完整的编码数据，导致对应的宏块（Macroblock）无法正确解码。为了避免画面完全损坏，解码器通常采用**错误隐藏（Error Concealment）**技术，用相邻区域或历史帧的信息填充缺失部分——本实验用统一的灰色块来代表这种「用空白填充」的最简单隐藏方式。丢包率越高，受损的块越多，画面呈现出越多的灰色方块。

### 抖动（Jitter）

抖动指数据包到达接收端的时间间隔不均匀（本应等间隔到达的包出现忽快忽慢的现象）。对于缺乏足够缓冲的实时直播场景，抖动会导致某些帧的数据未能按时到达而不完整，表现为画面出现**水平方向的行错位（撕裂，Tearing）**——本实验用逐行随机水平偏移来模拟这种效果。抖动幅度越大，行错位越明显。

### 带宽骤降（Bandwidth Drop）

当可用网络带宽突然不足以支撑当前码率时，编码器/传输系统被迫降低实际传输的分辨率或码率，接收端只能基于降采样后的低分辨率数据重建画面，表现为**整体模糊**——本实验用「先降采样、再放大」的方式模拟这种分辨率损失后的模糊效果。带宽下降幅度越大，画面越模糊。

### 三种损伤与网络条件的关系

| 损伤类型 | 主要触发条件 | 典型视觉特征 |
|---------|-------------|-------------|
| 丢包 | 网络拥塞、路由丢包、无线信道误码 | 局部灰色方块 |
| 抖动 | 网络路径切换、排队延迟波动 | 水平行错位/撕裂 |
| 带宽骤降 | 可用带宽持续不足 | 整体模糊 |

三种损伤在真实网络中往往**同时发生、相互叠加**，共同决定了用户最终感知到的视频质量。

## 实验设计

**合成测试帧**：540×960 分辨率，包含彩色条纹（用于测试色彩准确性）、渐变区域（测试低频响应）、密集细条纹（测试高频细节保真度），这三类内容分别对应视频画面中最常见的几种典型区域类型。

**三种损伤模拟函数**：`packet_loss()` 按块随机丢弃并填充灰色；`jitter()` 对每行做随机幅度的水平滚动位移；`bw_drop()` 使用 SciPy 的 `ndimage.zoom` 先降采样后放大。

**实验矩阵**：对每种损伤设置 4 个递增的强度级别，与原图一起构成 3×5 的全局对比网格；随后针对高强度损伤，额外提供差异热力图和像素级放大两种更细粒度的分析视角。

**真实感场景验证**：在合成测试帧上得到的结论，进一步在一个模拟自然风景 + 建筑 + 文字叠加（类似新闻画面）的真实感场景上重新验证，确认结论具有普适性而非仅适用于人工合成图案。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（SciPy 缩放在大尺寸帧上运算稍慢） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | NumPy、Matplotlib、SciPy（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
print('Imports OK')


## 步骤一：生成合成 HD 测试帧

`create_test_frame()` 构建了一张 540×960 的合成测试帧，特意划分为三个区域：

- **顶部彩色条**：多种纯色并排，用于测试损伤对色彩准确性的影响；
- **中部渐变区域**：从左到右的颜色渐变，代表低频、平滑变化的画面内容；
- **底部密集细条纹**：黑白相间的高频纹理，是压缩和传输损伤最容易暴露问题的区域。

这种分区设计能让我们在同一张图上，同时观察三种损伤对不同类型画面内容的影响差异。

In [ ]:
# Create synthetic HD test frame (scaled to 540x960)
H, W = 540, 960

def create_test_frame():
    frame = np.zeros((H, W, 3), dtype=np.float32)
    colors = [(1,0,0),(0,1,0),(0,0,1),(1,1,0),(0,1,1),(1,0,1),(1,1,1)]
    bar_w = W // len(colors)
    for i, c in enumerate(colors):
        frame[:H//3, i*bar_w:(i+1)*bar_w] = c
    for x in range(W):
        frame[H//3:2*H//3, x] = (x/W, 0.5, 1-x/W)
    for y in range(2*H//3, H, 20):
        for x in range(0, W, 4):
            v = 0.9 if (x//4+y//20)%2==0 else 0.2
            frame[y:y+10, x:x+2] = v
    return frame

original = create_test_frame()
fig, ax = plt.subplots(figsize=(12,7))
ax.imshow(np.clip(original,0,1))
ax.set_title('Original HD Test Frame (540x960)')
ax.axis('off')
plt.show()


## 步骤二：定义三种网络损伤模拟函数

这部分代码实现了三个损伤模拟函数，均已适配帧尺寸（frame-size-aware）：

- **`packet_loss(frame, rate)`**：把画面划分为若干 30×40 像素的小块，以概率 `rate` 随机选择部分块并填充为统一的灰色值，模拟错误隐藏后残留的「丢包块」；
- **`jitter(frame, px)`**：对画面的每一行，生成一个服从正态分布（标准差为 `px`）的随机整数位移，并沿水平方向滚动该行像素，模拟因抖动导致的行错位；
- **`bw_drop(frame, scale)`**：使用 SciPy 的 `ndimage.zoom` 先将画面按比例 `scale` 缩小，再放大回原始尺寸，模拟因带宽不足被迫降低分辨率后的模糊效果。

> 提示：三个函数的参数（`rate` / `px` / `scale`）分别控制各自损伤的严重程度，数值越极端（丢包率越高、抖动像素越大、缩放比例越小），损伤越严重。

In [ ]:
# Network impairment simulators (frame-size-aware)
def packet_loss(frame, rate):
    result = frame.copy()
    Hf, Wf = frame.shape[:2]
    bh, bw = 30, 40
    nb_h, nb_w = Hf//bh, Wf//bw
    mask = np.random.random((nb_h, nb_w)) < rate
    for i in range(nb_h):
        for j in range(nb_w):
            if mask[i,j]:
                i1, i2 = i*bh, min((i+1)*bh, Hf)
                j1, j2 = j*bw, min((j+1)*bw, Wf)
                result[i1:i2, j1:j2] = 0.5
    return result

def jitter(frame, px):
    result = frame.copy()
    Hf = frame.shape[0]
    shifts = (np.random.randn(Hf)*px).astype(int)
    for y in range(Hf):
        result[y] = np.roll(result[y], shifts[y], axis=0)
    return result

def bw_drop(frame, scale):
    Hf, Wf = frame.shape[:2]
    h2, w2 = int(Hf*scale), int(Wf*scale)
    low = ndimage.zoom(frame, (scale,scale,1), order=1)
    up = ndimage.zoom(low, (1/scale,1/scale,1), order=1)
    return np.clip(up[:Hf,:Wf], 0, 1)

print("Simulators ready (frame-size-aware)")


## 步骤三：全局对比网格——观察损伤强度递增的变化趋势

下方代码构建一个 3×5 的对比网格：每一行对应一种损伤类型（丢包/抖动/带宽骤降），每一列对应一个递增的强度级别（外加最左侧的原始画面作为参照）。

**观察重点**：
- 沿每一行从左到右浏览，观察该类损伤如何随强度增加而逐渐恶化；
- 对比三行在相同强度递增趋势下的视觉表现差异——丢包呈现离散的块状痕迹，抖动呈现连续的行错位，带宽骤降呈现整体渐进式模糊。

In [ ]:
# Full comparison grid
np.random.seed(42)
loss_rates = [0.01, 0.05, 0.10, 0.20]
jitter_px = [1, 3, 6, 10]
bw_scales = [0.75, 0.50, 0.25, 0.10]

fig, axes = plt.subplots(3, 5, figsize=(18, 12))

for row, (label, levels, func) in enumerate([
    ('Packet Loss', loss_rates, lambda f,l: packet_loss(f,l)),
    ('Jitter', jitter_px, lambda f,l: jitter(f,l)),
    ('BW Drop', bw_scales, lambda f,l: bw_drop(f,l)),
]):
    axes[row,0].imshow(original)
    axes[row,0].set_title('Original', fontweight='bold')
    for j, lv in enumerate(levels):
        imp = func(original, lv)
        axes[row,j+1].imshow(np.clip(imp,0,1))
        axes[row,j+1].set_title('{} {}'.format(label, lv))
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Network Impairments on HD Video', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 步骤四：差异热力图与像素级放大——定量观察受损区域

上一步的全局对比帮助你建立整体印象，这一步则提供更精细的分析工具，聚焦画面中细条纹纹理最密集的区域（高频细节最丰富、最容易暴露损伤的位置）：

- **第一行（Images）**：展示原图与三种「高强度损伤」场景（丢包 20%、抖动 20px、带宽骤降至 15%）下的画面；
- **第二行（差异热力图）**：计算受损图像与原图的逐像素差值并放大 5 倍显示（`hot` 配色，越亮代表差异越大），可以精确定位每种损伤实际影响的空间范围；
- **第三行（像素级放大）**：截取一小块 20×20 像素的区域并以最近邻插值放大展示，让你看清单个像素级别的破坏细节。

**观察重点**：丢包的热力图呈现清晰的矩形块状高亮；抖动的热力图沿水平方向呈现条纹状高亮；带宽骤降的热力图则是弥散、无明显边界的大范围中等亮度。

In [ ]:
# 局部放大对比：高损伤 + 差异热力图，让效果一目了然
# 选取底部细条纹理区域（高频细节最多的地方）
dy, dx = slice(340, 440), slice(60, 260)
ref = original[dy, dx]

# 三组高损伤场景
scenarios = [
    ('Packet Loss 20%', np.clip(packet_loss(original, 0.20)[dy, dx], 0, 1)),
    ('Jitter 20px', np.clip(jitter(original, 20)[dy, dx], 0, 1)),
    ('BW Drop 15%', np.clip(bw_drop(original, 0.15)[dy, dx], 0, 1)),
]

fig, axes = plt.subplots(3, 4, figsize=(18, 14))

# Row 0: Images
axes[0, 0].imshow(ref)
axes[0, 0].set_title('Original (Reference)', fontsize=12, fontweight='bold')
for j, (label, img) in enumerate(scenarios):
    axes[0, j+1].imshow(img)
    axes[0, j+1].set_title(label, fontsize=12, fontweight='bold')

# Row 1: Difference maps (amplified 5x, hot colormap)
axes[1, 0].axis('off')
for j, (label, img) in enumerate(scenarios):
    diff = np.abs(ref - img) * 5  # amplify for visibility
    im = axes[1, j+1].imshow(np.clip(diff, 0, 1), cmap='hot')
    axes[1, j+1].set_title(label + ' Error x5', fontsize=11)
    plt.colorbar(im, ax=axes[1, j+1], fraction=0.046)

# Row 2: Zoom into a 40x40 patch for pixel-level detail
px, py = 30, 120  # pick a spot with fine detail
sz = 20
axes[2, 0].imshow(ref[px:px+sz, py:py+sz], interpolation='nearest')
axes[2, 0].set_title('Original (pixel-level)', fontsize=11)
for j, (label, img) in enumerate(scenarios):
    axes[2, j+1].imshow(img[px:px+sz, py:py+sz], interpolation='nearest')
    axes[2, j+1].set_title(label + ' (pixel)', fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Network Impairments: Three-Level Detail Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n三重视角对比网络损伤：")
print("第一行：原始图像——丢包出现灰色方块，抖动导致水平错位，BW下降导致模糊")
print("第二行：差异热力图（差异x5）——越亮=损伤越重，可清晰定位受损区域")
print("第三行：像素级放大——可以看到单个像素被如何破坏")


## 步骤五：在真实感场景上验证结论

前面的实验都基于人工合成的测试图案，为了确认结论具有普适性而非仅是合成图案的特例，这一步构建了一个更接近真实视频内容的场景：`create_realistic_scene()` 生成包含天空渐变、远山轮廓、地面纹理、多栋建筑（含窗户细节）以及底部文字叠加条（模拟新闻字幕栏）的合成画面。

随后对这个真实感场景应用三种网络损伤（各取一个中等强度），并排展示对比结果。

**观察重点**：即使画面内容从人工图案变为更自然的场景，三种损伤各自的视觉特征（灰色块/行错位/整体模糊）依然清晰可辨，验证了实验结论并非合成图案的偶然产物。

In [ ]:
# 模拟真实视频场景：自然风景 + 文字叠加（类新闻画面）
def create_realistic_scene():
    H, W = 360, 640
    frame = np.zeros((H, W, 3), dtype=np.float32)
    # 天空渐变
    for y in range(H//2):
        t = y / (H//2)
        frame[y, :] = (0.3+0.4*t, 0.4+0.3*t, 0.6+0.3*t)
    # 远山轮廓
    import random; random.seed(42)
    for x in range(W):
        h = int(H*0.35 + np.sin(x*0.01)*20 + np.sin(x*0.03)*15)
        frame[h:H//2+30, x] = (0.15, 0.35, 0.15)
    # 地面纹理
    for y in range(H//2+30, H):
        v = 0.2 + 0.15 * np.sin(y*0.05) * np.sin(y*0.02)
        frame[y, :] = (v, 0.25+v*0.3, v*0.5)
    # 建筑群
    buildings = [(50,80,120,0.6),(180,50,200,0.5),(320,100,150,0.45),(450,70,170,0.55)]
    for bx, bw, bh, c in buildings:
        top = H//2+30 - bh
        frame[top:H//2+30, bx:bx+bw] = (c*0.7, c*0.6, c*0.5)
        # 窗户
        for wy in range(top+5, top+bh-5, 15):
            for wx in range(bx+5, bx+bw-5, 12):
                frame[wy:wy+8, wx:wx+6] = (0.8, 0.85, 0.7) if (wx+wy)%30<15 else (0.3, 0.3, 0.2)
    # 文字叠加（模拟新闻标题栏）
    frame[H-40:H-10, 20:W-20] = (0.05, 0.05, 0.1)
    frame[H-35:H-15, 30:W-30] = (0.9, 0.9, 0.9)
    return frame

scene = create_realistic_scene()
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(np.clip(scene, 0, 1))
ax.set_title('Simulated Video Scene (Landscape + Buildings + Text Overlay)')
ax.axis('off')
plt.show()


In [ ]:
# 在真实感场景上应用网络损伤
np.random.seed(123)
impairments_real = [
    ('Original', lambda f: f),
    ('Loss 10%', lambda f: packet_loss(f, 0.10)),
    ('Jitter 6px', lambda f: jitter(f, 6)),
    ('BW Drop 30%', lambda f: bw_drop(f, 0.30)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for j, (label, func) in enumerate(impairments_real):
    imp = func(scene)
    axes[j].imshow(np.clip(imp, 0, 1))
    axes[j].set_title(label, fontsize=12, fontweight='bold')
    axes[j].axis('off')
plt.suptitle('Realistic Scene: Network Impairment Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n在实际视频传输中，网络损伤的影响体现在：")
print("1. 丢包 -> 画面中出现灰色方块（错误隐藏后的残留）")
print("2. 抖动 -> 画面行错位，在快速运动场景中最明显")
print("3. 带宽下降 -> 整体清晰度降低，细节丢失")
print("4. 真实系统中三种损伤往往同时发生，互相叠加")


## 实验结果与分析

### 三种损伤的独特视觉特征

综合合成测试帧与真实感场景两组实验，可以总结出三种网络损伤各自稳定、可辨识的视觉特征：

- **丢包**：呈现离散、边界清晰的矩形灰色块，随丢包率增加，受损块的数量增多但每个块的形态基本不变；
- **抖动**：呈现连续的水平行错位（撕裂感），在包含垂直细节（如建筑物竖直边缘）的区域最为明显，随抖动幅度增加，错位程度加剧；
- **带宽骤降**：呈现均匀的整体模糊，没有局部突兀的痕迹，随降采样比例增大，模糊程度逐渐加重，细节纹理最先消失。

### 合成图案 vs 真实感场景的差异

在合成图案中，密集细条纹区域对三种损伤都表现得最为敏感——这是因为高频内容一旦丢失就难以从周围信息推断恢复。在真实感场景中，建筑物的窗户细节和文字叠加条同样是最先出现明显失真的区域，这与「高频细节最先受损」的结论一致，进一步验证了合成图案的测试设计具有实际代表性。

### 组合损伤效果

真实网络环境中，丢包、抖动、带宽骤降往往同时发生。虽然本实验主要单独展示每种损伤，但从视觉特征的独立性可以推断：三种损伤同时出现时，画面会同时呈现「局部灰块 + 局部撕裂 + 整体模糊」的复合特征，对观感的破坏程度通常比单一损伤更严重。

## 从实验到实际系统

本实验演示了单一损伤类型的独立视觉特征，真实生产系统采用了多种机制来缓解这些问题：

- **YouTube 等平台的 ABR + FEC**：自适应码率（ABR）根据实时带宽调整码率避免因带宽不足产生骤降式模糊；前向纠错（Forward Error Correction, FEC）通过冗余数据主动恢复部分丢失的数据包，减少丢包对画面的影响；
- **WebRTC 的 NACK 机制**：通过否定确认（Negative Acknowledgment, NACK）请求重传丢失的关键帧数据，同时结合抖动缓冲区（Jitter Buffer）平滑数据包到达时间的波动；
- **5G URLLC（超可靠低时延通信）**：针对对时延和可靠性要求极高的场景（如远程手术、工业控制），从网络层面降低丢包率和抖动幅度；
- **为什么实验七的 VR 场景更严苛**：本实验的损伤模拟方法将直接应用于下一个实验中的 VR/360 全景视频，你会看到相同强度的损伤在 VR 场景下造成的感知影响远比本实验的 HD 场景更严重——这与 VR 视频独特的等距矩形投影和视场角机制密切相关。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **三种网络损伤各有独特的视觉特征**：丢包呈块状，抖动呈行错位，带宽骤降呈整体模糊；
2. **高频细节区域对损伤最敏感**：无论是合成图案还是真实场景，细节丰富的区域都是最先暴露失真的位置；
3. **差异热力图和像素级放大是量化分析的有效工具**：能帮助从主观观察进一步转向客观定位；
4. **合成测试与真实场景结论一致**：证明了人工设计的测试图案具有实际代表性；
5. **真实系统通过多种机制协同缓解损伤**：ABR、FEC、抖动缓冲区等技术分别针对不同损伤类型发挥作用。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **修改丢包率**：将 `loss_rates` 中的数值调整为更极端的范围（如 0.3、0.5），观察画面是否出现无法辨认的临界点。
2. **改变块大小**：修改 `packet_loss()` 中的 `bh, bw` 参数（如从 30×40 改为 60×80），观察块大小对视觉严重程度的影响。
3. **对比不同场景**：将本实验用于真实感场景的损伤强度参数应用到合成测试帧上（或反过来），比较两种画面内容对相同损伤强度的敏感度差异。
4. **组合多种损伤**：编写代码同时应用丢包 + 带宽骤降（例如先调用 `bw_drop()` 再调用 `packet_loss()`），观察复合损伤的视觉效果是否比单一损伤更严重。
5. **量化 PSNR**：复用实验三中的 `psnr()` 函数，计算三种损伤在不同强度下对应的 PSNR 数值，将主观视觉观察转化为客观量化指标。

---

← [实验五：语义通信 vs 传统传输](https://www.kaggle.com/code/guopingtan/fmi-demo5-semantic-communication) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验七：网络损伤对 VR/360 视频的影响 →](https://www.kaggle.com/code/guopingtan/fmi-demo-7-network-impairments-on-vr-360-video)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University